In [2]:
# On your own machine, install PyTorch + torchvision for your hardware first (pytorch.org)
!git clone https://github.com/SelsabeelA/Invisible_Gorilla_Effect_Reproduction.git
%cd Invisible_Gorilla_Effect_Reproduction
!pip install -q -r requirements.txt isic-cli

Cloning into 'Invisible_Gorilla_Effect_Reproduction'...
remote: Enumerating objects: 384, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 384 (delta 0), reused 1 (delta 0), pack-reused 378 (from 4)
Receiving objects: 100% (384/384), 69.43 MiB | 27.84 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/kaggle/working/Invisible_Gorilla_Effect_Reproduction


In [9]:
import os, re, sys, random, shutil, subprocess, tempfile, importlib.util
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

SEED, EPOCHS, BATCH = 1, 30, 16
ISIC, INK = "data/ISIC/", "data/ISIC/annotations/Ink_annotations/"
CKPT = f"outputs/saved_models/ISIC/ResNet18_ISIC-{SEED}.pth"
ISIC_CMD = [sys.executable, "-c", "from isic_cli.cli import main; main()"]
EV = "evaluate_OOD_detection_method.py"

# 30 epochs instead of the repo's 600, and a trailing slash so image paths join correctly
s = open("source/config/ISIC.py").read()
s = re.sub(r"^num_epochs = \d+", f"num_epochs = {EPOCHS}", s, flags=re.M)
s = re.sub(r"^loader_root = 'data/ISIC/?'", "loader_root = 'data/ISIC/'", s, flags=re.M)
open("source/config/ISIC.py", "w").write(s)

# the eval function returns [[name, AUROC, AUCPR]] but the script unpacks two values
lines = open(EV).read().split("\n")
for i, l in enumerate(lines):
    if "evaluate_ood_detection_method(args.method" in l and "return_metrics=True" in l:
        lines[i] = l[: len(l) - len(l.lstrip())] + ("_, AUROC, AUCPR = evaluate_ood_detection_method("
                   "args.method,net,ID_loader,OOD_loader,return_metrics=True,**kwargs_test)[0]")
open(EV, "w").write("\n".join(lines))

9278

In [10]:
for cid, d in [(74, "2016/train"), (61, "2016/test")]:
    if not os.path.exists(f"{ISIC}{d}/metadata.csv"):
        subprocess.run(ISIC_CMD + ["image", "download", "--collections", str(cid), d], cwd=ISIC, check=True)

from source.config import ISIC as cf
if not os.path.exists(ISIC + "train.csv"):
    parts = []
    for split in ["train", "test"]:
        m = pd.read_csv(f"{ISIC}2016/{split}/metadata.csv")
        m["Path"] = f"2016/{split}/" + m["isic_id"] + ".jpg"
        parts.append(m)
    meta = pd.concat(parts, ignore_index=True)
    lab = meta["diagnosis_1"].astype(str).str.lower()
    meta = meta[lab.isin(["benign", "malignant"])].copy()
    meta["malignant"] = (lab[meta.index] == "malignant").astype(int)
    meta["benign"] = 1 - meta["malignant"]
    meta["patient_id"] = meta["isic_id"]
    out = meta[["Path", "malignant", "benign", "patient_id"]].sample(frac=1, random_state=0)
    for c in dict.fromkeys(cf.classes):   # the repo's code expects a column per class
        if c not in out.columns:
            out[c] = 0
    n = int(0.8 * len(out))
    out.iloc[:n].to_csv(ISIC + "train.csv", index=False)
    out.iloc[n:].to_csv(ISIC + "valid.csv", index=False)

If you have been granted special permissions, logging in with `isic user login` might return more data.




Successfully downloaded 900 images to 2016/train/.
Successfully wrote 900 metadata records to 2016/train/metadata.csv.
Successfully wrote attributions to 2016/train/attribution.txt.
Successfully wrote 1 license(s) to 2016/train/licenses.


If you have been granted special permissions, logging in with `isic user login` might return more data.




Successfully downloaded 379 images to 2016/test/.
Successfully wrote 379 metadata records to 2016/test/metadata.csv.
Successfully wrote attributions to 2016/test/attribution.txt.
Successfully wrote 1 license(s) to 2016/test/licenses.


In [11]:
norm = lambda p: re.sub(r"^(\.\./)*data/ISIC_collection_processed/", "", p.strip().strip(","))
colour = {c: [norm(l) for l in open(f"{INK}{c}.txt") if l.strip()] for c in ["red", "black"]}
random.seed(0)
targets = colour["red"] + random.sample(colour["black"], 75)

def fetch(rel):
    dest = ISIC + rel
    if os.path.exists(dest):
        return None
    isic_id = os.path.basename(rel)[:-4]
    with tempfile.TemporaryDirectory() as t:
        subprocess.run(ISIC_CMD + ["image", "download", "--search", f"isic_id:{isic_id}", t],
                       capture_output=True, text=True)
        jpg = os.path.join(t, isic_id + ".jpg")
        if not os.path.exists(jpg):
            return None
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.move(jpg, dest)
        m = pd.read_csv(os.path.join(t, "metadata.csv"))
        m["Path"] = rel
        return m

with ThreadPoolExecutor(4) as ex:
    got = [m for m in ex.map(fetch, targets) if m is not None]
print(f"newly downloaded: {len(got)} of {len(targets)}")

if got:  # added to valid.csv; they're outside the training list, so they only act as OOD images
    new = pd.concat(got, ignore_index=True)
    lab = new["diagnosis_1"].astype(str).str.lower()
    keep = new[lab.isin(["benign", "malignant"])].copy()
    keep["malignant"] = (lab[keep.index] == "malignant").astype(int)
    keep["benign"] = 1 - keep["malignant"]
    keep["patient_id"] = keep["isic_id"]
    train, valid = pd.read_csv(ISIC + "train.csv"), pd.read_csv(ISIC + "valid.csv")
    add = keep[~keep.Path.isin(set(train.Path) | set(valid.Path))].copy()
    for c in valid.columns:
        if c not in add.columns:
            add[c] = 0
    pd.concat([valid, add[valid.columns]], ignore_index=True).to_csv(ISIC + "valid.csv", index=False)

newly downloaded: 98 of 100


In [12]:
if not os.path.exists(CKPT):
    !python training.py --dataset ISIC --setting setting1 --net_type ResNet18 --seed {SEED} --batch_size {BATCH} --save_path outputs/saved_models

There are 264 images with malignant
There are 1103 images with benign
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

Experiment details:
| Running experiment seed: 1
| Classes ID: ['malignant', 'benign']
| Length of train dataset: 620
| Length of validation dataset: 155

Training model
| Training Epochs = 30
| Initial Learning Rate = 0.001
| Optimiser = SGD
| Scheduler = MultiStepLR
| Batch size = 16

=> Training Epoch [  0/ 30], LR=0.0010
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 work

In [ ]:
ink_file = INK + "ink_annotation.txt"
if not os.path.exists(ink_file + ".orig"):
    shutil.copy(ink_file, ink_file + ".orig")
methods = {"MCP": [], "ODIN": [], "mahalanobis": ["--mahalanobis_module", "-1"], "KNN": [],
           "ReAct": [], "ViM": [], "KDE": [], "LOF": [], "gradnorm": [], "SHE": []}
pat = re.compile(r"(\S+) AUROC: ([-+0-9.eE]+) AUCPR: ([-+0-9.eE]+)")
rows = []
try:
    for col in ["red", "black"]:
        open(ink_file, "w").write("\n".join(colour[col]) + "\n")   # the loader reads one hardcoded ink list
        for m, extra in methods.items():
            r = subprocess.run([sys.executable, EV, "--seed", str(SEED), "--method", m,
                                "--ood_type", "different_class", "--setting", "setting1",
                                "--batch_size", str(BATCH), "--return_metrics", "True", *extra],
                               capture_output=True, text=True)
            hit = pat.findall(r.stdout)
            if hit:
                rows.append({"colour": col, "method": m, "AUROC": float(hit[-1][1]), "AUCPR": float(hit[-1][2])})
            else:
                print("FAILED", col, m, "->", (r.stderr.strip().splitlines() or ["?"])[-1])
finally:
    shutil.copy(ink_file + ".orig", ink_file)

res = pd.DataFrame(rows)
res.to_csv("ood_results.csv", index=False)
pd.DataFrame({"method": res.method, "artefact": "ink",
              "similar/dissimilar": res.colour.map({"red": "similar", "black": "dissimilar"}),
              "fold": 0, "AUROC": res.AUROC, "AURC": "", "FPR@TPR80": ""}).to_csv("results_partI.csv", index=False)
print(res.pivot(index="method", columns="colour", values="AUROC"))